# 🔍 01 · Exploración del dataset

**Objetivo:** conocer a fondo los datos *antes* de limpiarlos. Aquí no hacemos gráficos bonitos todavía;
hacemos de **detectives**: ¿qué tipo tiene cada columna? ¿hay datos faltantes? ¿hay columnas inútiles?

> 💡 Regla de oro del EDA: **nunca confíes en los datos hasta haberlos interrogado.** Un dataset que
> *parece* limpio casi siempre esconde sorpresas.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.data_loading import load_raw

pd.set_option("display.max_columns", 40)
df = load_raw()
print("Dataset cargado:", df.shape)

Dataset cargado: (8910, 38)


## 1. Radiografía general con `.info()`

`.info()` nos dice, por columna: cuántos valores **no nulos** tiene y de qué **tipo** es
(`object` = texto, `float64`/`int64` = número). Es la primera foto de la "salud" del dataset.

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8910 entries, 0 to 8909
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   strain_name            8910 non-null   object 
 1   breeder                8910 non-null   object 
 2   description            8910 non-null   object 
 3   current_price_gbp      8910 non-null   float64
 4   original_price_gbp     8910 non-null   float64
 5   discount_percent       8910 non-null   float64
 6   pack_options           8910 non-null   object 
 7   product_url            8910 non-null   object 
 8   overview               8910 non-null   object 
 9   growth_and_harvest     8910 non-null   object 
 10  experience             8910 non-null   object 
 11  seed_type              8910 non-null   object 
 12  flowering_period_type  8910 non-null   object 
 13  indica_sativa          8910 non-null   object 
 14  type_ratio             8910 non-null   object 
 15  stra

### 🧠 Teoría: valores nulos (faltantes)

Un **valor nulo** (`NaN`) es un dato que falta. Suelen ser un dolor de cabeza porque muchos cálculos
fallan o se sesgan con ellos. Contémoslos en todo el dataset:

In [3]:
total_nulos = df.isna().sum().sum()
print(f"Total de valores nulos en TODO el dataset: {total_nulos}")

Total de valores nulos en TODO el dataset: 0


## 2. El mito de los "0% nulos" ⚠️

El dataset dice tener **0 nulos**. Suena perfecto... pero es una trampa típica de los datos **scrapeados**
(extraídos automáticamente de una web): en vez de dejar celdas vacías, el scraper las rellenó con
**valores repetidos o plantillas**. Es "basura disfrazada de dato".

> **¿Por qué un scraper rellena con plantillas en vez de dejar vacío?** Porque suele leer una plantilla de
> la página web (un texto fijo que el sitio muestra siempre) o pone un valor por defecto cuando no encuentra
> el dato. El resultado: la celda *tiene* contenido, así que `isna()` no la detecta como faltante, aunque en
> la práctica no aporte información. Por eso hay que buscar la basura de otra forma.

Para descubrirla usamos la **cardinalidad**: el número de valores **distintos** que tiene cada columna
(`.nunique()`). Una columna con **un solo** valor distinto no aporta *ninguna* información.

In [4]:
# Ordenamos las columnas de menos a más valores distintos
cardinalidad = df.nunique().sort_values()
cardinalidad

sale_item                   1
most_popular_seeds          1
stock_availability          2
flowering_period_type       3
seed_type                   3
discount_percent            5
strength                    6
indoor_flowering_time       7
growth_and_harvest          7
environment                 9
indoor_height_detail        9
indica_sativa              15
climate                    26
strain_type_summary        36
outdoor_harvest_time       42
experience                 52
harvest_month             128
cbd                       226
height_indoor             227
breeder                   268
genetic_background        271
height_outdoor            295
original_price_gbp        404
flavor                    445
type_ratio                448
yield_indoor              469
yield_outdoor             626
flowering_time            690
thc                       804
effect                   1116
medical_strains          1419
current_price_gbp        1776
pack_options             2385
smell_tast

### 🕵️ Columnas constantes (1 solo valor = inútiles)

Filtramos las columnas cuya cardinalidad es exactamente 1. Esas se pueden borrar sin perder nada.

In [5]:
n_unique = df.nunique()
columnas_constantes = n_unique[n_unique == 1]
print("Columnas con un solo valor (basura):")
columnas_constantes

Columnas con un solo valor (basura):


sale_item             1
most_popular_seeds    1
dtype: int64

### El otro extremo: columnas con MUchísima variedad

Miremos ahora las 5 columnas con **más** valores distintos. Esto también informa: nos dice cuáles son
identificadores o texto libre (mucha variedad) y cuáles son categorías (poca variedad).

In [6]:
df.nunique().sort_values(ascending=False).head()

description    8524
strain_name    8080
product_url    4253
overview       3552
smell_taste    2899
dtype: int64

Tiene sentido: `strain_name`, `description`, `overview` y `product_url` son **texto libre o casi único**
por cada cepa (por eso tienen tanta variedad). No sirven para agrupar, pero `description` sí servirá más
adelante para una nube de palabras.

## 3. Columnas "plantilladas" (casi-constantes)

Otras columnas no tienen 1 valor, pero sí muy pocos y todos son frases genéricas idénticas. Veamos dos
ejemplos claros, `experience` y `growth_and_harvest`:

In [7]:
df['experience'].value_counts().head()

experience
Users report a balanced and enjoyable experience with this strain.                                                                                                                                                                                                                                                                                                            8857
Super Silver Haze Autoflowering delivers a mentally uplifting high that encourages creativity and happiness, characterized by a smooth, sweet, and spicy taste with citrus notes. It's potent, quick to grow, and suits both experienced and moderately novice growers.                                                                                                          2
Despite its indica dominance, Somango XXL offers balanced effects that are uplifting and creative, perfect for starting the day. Its flavor profile includes a mix of fresh mango, floral notes, and fruity sweetness, making it appealing for recreati

In [8]:
df['growth_and_harvest'].value_counts()

growth_and_harvest
This strain grows well in both indoor and outdoor environments with proper care and nutrients.                                                                                                                                                                                                                                                                                                                              8904
The strain flourishes under a standard 12-12 light regime, maturing in 9–10 weeks with significant colas expansion reminiscent of OG Kush but surprises with fruity aromas. HulkBerry excels in both SOG and ScrOG setups, promising bountiful yields that can impress even seasoned Kush cultivators.                                                                                                                         1
This strain is capable of yielding two all-female crops outdoors in one season under warm, sunny conditions. Indoors, it transitions to flowering a

Como ves, son frases plantilladas ("Users report a balanced and enjoyable experience...",
"This strain grows well in both indoor and outdoor...") que se repiten miles de veces. `growth_and_harvest`
tiene solo 7 frases distintas para 8.910 cepas. Tampoco sirven para analizar: en el notebook `02` las
eliminaremos junto con las constantes.

## 4. Números escondidos como texto

Estas columnas *deberían* ser números, pero vienen como texto porque incluyen el símbolo o la unidad.
Fíjate en los valores:

In [9]:
df[['thc', 'cbd', 'flowering_time', 'yield_indoor']].head()

,thc,cbd,flowering_time,yield_indoor
0,21.8%,1.5%,64 days,406.0g/m²
1,20.9%,1.0%,90 days,581.0g/m²
2,24.2%,1.9%,60 days,514.0g/m²
3,20.4%,1.6%,67 days,606.0g/m²
4,23.0%,1.5%,65 days,599.0g/m²


`'21.8%'`, `'64 days'`, `'406.0g/m²'`... son **texto**, no números. Python no puede promediar texto.
En el notebook `02` los convertiremos a número (ya tenemos las funciones listas en `src/data_loading.py`).

## 5. Columnas multi-etiqueta

Algunas columnas guardan **varias cosas en una sola celda**, separadas por comas:

In [10]:
df[['effect', 'flavor', 'medical_strains']].head()

,effect,flavor,medical_strains
0,"Relaxed, Energetic",Citrus,"Appetite Stimulation, Nausea, Anxiety"
1,"Energetic, Relaxed","Sweet, Citrus, Pine","Insomnia, Pain Relief"
2,"Euphoric, Focused","Earthy, Pine","Appetite Stimulant, Social"
3,Creative,"Berry, Earthy, Pine",Euphoric
4,"Creative, Euphoric",Earthy,"Appetite Stimulant, Body Stone, Cerebral, Crea..."


Una celda como `'Relaxed, Energetic'` contiene **dos** efectos. Para contar "¿cuál es el efecto más
común?" tendremos que **separar** (explotar) esas listas. También lo haremos en el notebook `02`.

## 6. Nombres de cepa repetidos

`strain_name` tiene 8.080 valores únicos entre 8.910 filas. Eso significa que hay **nombres repetidos**.
Veamos los más frecuentes:

In [11]:
df['strain_name'].value_counts().head(10)

strain_name
White Widow                27
Girl Scout Cookies         18
Wedding Cake               17
OG Kush                    17
Blue Dream                 11
Girl Scout Cookies Auto    11
Sour Diesel                10
Zkittlez                   10
Bruce Banner               10
Gorilla Auto                9
Name: count, dtype: int64

Hay cepas con el mismo nombre varias veces. Normalmente son la **misma cepa vendida por distintos
breeders** o en distintas presentaciones. No son necesariamente errores; solo hay que tenerlo presente si
más adelante contamos "cepas únicas".

---
### 📝 Resumen de la exploración

| Hallazgo | Ejemplo | Qué haremos |
|---|---|---|
| Columnas constantes | `sale_item`, `most_popular_seeds` | Eliminar |
| Columnas plantilladas | `experience`, `growth_and_harvest` | Eliminar |
| Números como texto | `thc` = `'21.8%'` | Convertir a número |
| Multi-etiqueta | `effect` = `'Relaxed, Energetic'` | Separar (explode) |
| Tipo sucio | `indica_sativa` (15 variantes) | Consolidar a 3 categorías |
| Nombres repetidos | mismos nombres por varios breeders | Tener en cuenta al contar |

Con este mapa claro, el notebook `02_limpieza_features` tendrá un plan concreto, no limpieza a ciegas.